# 00_00 — Data Ingestion

**Pipeline:** `data_ingestion`  
**Kedro node:** `load_all_flights_node`  
**Output catalog:** `preprocessed_flights` → `data/02_intermediate/preprocessed/`

---

## Contexto

O dataset ALFA contém voos de uma aeronave de asa fixa (CarbonZ) com diferentes tipos de falha induzidas.
Cada voo está armazenado como uma pasta com múltiplos arquivos `.csv`, onde cada arquivo corresponde a
um **tópico ROS** (fonte de dados de um sensor ou subsistema).

Este notebook demonstra o fluxo da pipeline `data_ingestion`:
1. **Merge** de todos os tópicos de um voo em um único DataFrame alinhado no tempo (`merge_asof`)
2. **Filtragem** de colunas de ruído (metadados ROS, covariâncias, canais RC, etc.)

> **Para rodar toda a pipeline no Kedro:** `kedro run --pipeline=data_ingestion`

## Imports e parâmetros

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# Funções da pipeline — as mesmas que o Kedro executa em producao
from aeroespacial_2.pipelines.data_ingestion.nodes import (
    filter_noise_columns,
    merge_flight_csvs,
)

# Parâmetros espelhando conf/base/parameters.yml
FLIGHT_DIR = "../../data/01_raw/processed/carbonZ_2018-07-18-15-53-31_1_engine_failure"
IMU_SOURCE_TO_DISCARD = "imu-data_raw"

## Passo 1 — Merge dos tópicos ROS

`merge_flight_csvs` carrega cada `.csv` da pasta, prefixa as colunas com o nome do tópico,
normaliza o tempo para segundos e alinha tudo via `merge_asof` (backward fill).

O DataFrame base é o tópico mais denso (mais linhas), e os demais são interpolados temporalmente.

In [ ]:
df_merged = merge_flight_csvs(FLIGHT_DIR)

print(f"Shape após merge: {df_merged.shape}")
print(f"Tempo: {df_merged['%time'].min():.2f}s → {df_merged['%time'].max():.2f}s")
df_merged.head(3)

In [ ]:
# Fontes (topicos ROS) presentes no merge
fontes = sorted(set(
    col.split("_field.")[0].split("engine_failure-")[-1]
    for col in df_merged.columns
    if "_field." in col
))
print(f"{len(fontes)} fontes detectadas:")
for f in fontes:
    print(f"  - {f}")

## Visualização de um tópico

Função auxiliar para inspecionar qualquer tópico diretamente no DataFrame bruto.

In [ ]:
def plot_topic(df: pd.DataFrame, topic: str, skip_noise: bool = True) -> None:
    """Plota todas as colunas de um tópico ROS contra o tempo."""
    time_col = "%time"
    cols = [c for c in df.columns if topic in c and "_field." in c]
    if skip_noise:
        noise = ["covariance", "checksum", "seq", "stamp", "frame_id", "sysid", "compid"]
        cols = [c for c in cols if not any(k in c.lower() for k in noise)]
    if not cols:
        print(f"Nenhuma coluna para: {topic}")
        return
    fig, axes = plt.subplots(len(cols), 1, figsize=(14, 2.2 * len(cols)), sharex=True)
    if len(cols) == 1:
        axes = [axes]
    for ax, col in zip(axes, cols):
        label = col.split("_field.")[-1]
        ax.plot(df[time_col], df[col], color="tab:blue", linewidth=0.8)
        ax.set_ylabel(label, fontsize=8)
        ax.grid(True, alpha=0.3)
    axes[-1].set_xlabel("Tempo (s)")
    plt.suptitle(f"Tópico: {topic}", fontsize=13)
    plt.tight_layout()
    plt.show()

In [ ]:
# VFR HUD: instrumentos de voo consolidados (airspeed, altitude, throttle...)
plot_topic(df_merged, "mavros-vfr_hud")

In [ ]:
for topic in fontes:
    plot_topic(df_merged, topic)

## Passo 2 — Filtragem de colunas de ruído

`filter_noise_columns` remove:
- Metadados ROS: headers, covariâncias, checksums
- Tópicos irrelevantes: `diagnostics`, `mavlink-from`, IMU raw
- Campos redundantes: GPS absoluto, canais RC, flags de estado MAVLink

In [ ]:
df_preprocessed = filter_noise_columns(df_merged, imu_source_to_discard=IMU_SOURCE_TO_DISCARD)

print(f"Colunas antes:  {df_merged.shape[1]}")
print(f"Colunas depois: {df_preprocessed.shape[1]}")
print(f"Removidas:      {df_merged.shape[1] - df_preprocessed.shape[1]}")

In [ ]:
# Colunas mantidas (primeiras 20)
for col in df_preprocessed.columns[:20]:
    print(col)

## Resultado

O DataFrame `df_preprocessed` é o que a pipeline salva em `data/02_intermediate/preprocessed/`.
Pronto para ser consumido pela pipeline `data_preparation`.

---
**Próximo:** `01_00_data_exploration.ipynb` → limpeza, engenharia de features, preparação final.

In [ ]:
print(f"Shape final: {df_preprocessed.shape}")
df_preprocessed.head(3)

## Checagem — Colunas do `df_preprocessed`

Verificar se alguma coluna constante (ou com variação desprezível) sobreviveu à filtragem.  
Colunas assim não carregam informação útil e devem ser adicionadas à lista de filtros.

In [ ]:
numeric_cols = df_preprocessed.select_dtypes(include="number").columns.drop("%time")
std = df_preprocessed[numeric_cols].std().sort_values()

const_cols = std[std == 0].index.tolist()
near_const_cols = std[(std > 0) & (std < 1e-3)].index.tolist()

print(f"Colunas constantes (std = 0): {len(const_cols)}")
for c in const_cols:
    print(f"  {c.split('_field.')[-1]:<55}  valor={df_preprocessed[c].iloc[0]}")

print(f"\nColunas quasi-constantes (0 < std < 1e-3): {len(near_const_cols)}")
for c in near_const_cols:
    print(f"  {c.split('_field.')[-1]:<55}  std={std[c]:.2e}")

In [ ]:
fontes_preprocessed = sorted(set(
    col.split("_field.")[0].split("engine_failure-")[-1]
    for col in df_preprocessed.columns
    if "_field." in col
))

print(f"{len(fontes_preprocessed)} tópicos em df_preprocessed:")
for t in fontes_preprocessed:
    print(f"  - {t}")

for topic in fontes_preprocessed:
    plot_topic(df_preprocessed, topic, skip_noise=False)